In [ ]:
from google import genai

In [ ]:
#!pip install -q -U google-genai

#from google import genai
from google.genai import types

In [ ]:
from google.colab import userdata
GEMINI_API_KEY=userdata.get('geminikl')

In [ ]:
client=genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# Student budget
budget = 1000

# Expense history
expenses = []

print("Initial budget: ₹", budget)

Initial budget: ₹ 1000


In [ ]:
def add_expense(category: str, amount: float) -> str:
    """Record a completed expense in a category."""

    if amount <= 0:
        return "Expense amount must be greater than zero."

    if amount > budget - sum(e["amount"] for e in expenses):
        return "Insufficient balance. Expense was not recorded."

    expenses.append({
        "category": category,
        "amount": amount
    })

    return f"Recorded ₹{amount:.2f} for {category}."

In [ ]:
def check_balance() -> str:
    """Show the remaining student budget."""

    spent = sum(e["amount"] for e in expenses)
    balance = budget - spent

    return f"Budget: ₹{budget:.2f}, Spent: ₹{spent:.2f}, Remaining: ₹{balance:.2f}"

In [ ]:
def show_expenses() -> str:
    """Show all recorded student expenses."""

    if not expenses:
        return "No expenses recorded yet."

    result = ""

    for e in expenses:
        result += f'{e["category"]}: ₹{e["amount"]:.2f}\n'

    return result

In [ ]:
def check_affordability(amount: float) -> str:
    """Check whether the student can afford a proposed purchase."""

    remaining = budget - sum(e["amount"] for e in expenses)

    if amount <= 0:
        return "Amount must be greater than zero."

    if amount > remaining:
        return f"Not affordable. You have only ₹{remaining:.2f}."

    return f"Affordable. Balance after purchase would be ₹{remaining - amount:.2f}. This check does not record the expense."

In [ ]:
def category_summary() -> str:
    """Calculate total spending in each category."""

    if not expenses:
        return "No expenses available."

    totals = {}

    for e in expenses:
        category = e["category"]
        totals[category] = totals.get(category, 0) + e["amount"]

    return "\n".join(
        f"{category}: ₹{amount:.2f}"
        for category, amount in totals.items()
    )

In [ ]:
def biggest_expense_category() -> str:
    """Identify the category with the highest total spending."""

    if not expenses:
        return "No expenses available."

    totals = {}

    for e in expenses:
        category = e["category"]
        totals[category] = totals.get(category, 0) + e["amount"]

    category = max(totals, key=totals.get)

    return f"Highest spending category: {category}, ₹{totals[category]:.2f}"

In [ ]:
def saving_advice() -> str:
    """Give simple advice based on the remaining budget."""

    spent = sum(e["amount"] for e in expenses)
    remaining = budget - spent
    spent_percent = (spent / budget) * 100

    if remaining < 0:
        return "You are over budget. Stop non-essential spending."

    if spent_percent >= 80:
        return "You have used at least 80% of your budget. Avoid unnecessary purchases."

    if spent_percent >= 50:
        return "You have used at least half your budget. Plan your remaining expenses carefully."

    return "Your spending is within budget. Continue tracking expenses and save for emergencies."

In [ ]:
instructions = """
You are a friendly Student Expense Tracker Agent.

The student's total budget is ₹1000.

Understand natural English and select the appropriate tools.

Rules:
1. If the student says they have spent money, use add_expense.
2. If they ask whether they can afford something, use check_affordability.
3. If they ask for the balance, use check_balance.
4. If they ask for expense history, use show_expenses.
5. If they ask for category totals, use category_summary.
6. If they ask where they spend the most, use biggest_expense_category.
7. If they ask how to save money, use saving_advice.
8. Never record a proposed expense unless the student says to record it.
9. Never invent an expense amount or claim a tool succeeded if it failed.
10. Explain the result in simple English and use Indian rupees.
"""

chat = client.chats.create(
    model="gemini-3.6-flash",
    config=types.GenerateContentConfig(
        system_instruction=instructions,
        tools=[
            add_expense,
            check_balance,
            show_expenses,
            check_affordability,
            category_summary,
            biggest_expense_category,
            saving_advice
        ],
        temperature=0.2
    )
)

print("Student Expense Tracker Agent created!")

Student Expense Tracker Agent created!


In [ ]:
while True:
    message = input("\nStudent: ")

    if message.lower() in ["exit", "quit", "stop"]:
        print("Agent: Goodbye! Spend wisely.")
        break

    try:
        response = chat.send_message(message)
        print("Agent:", response.text)

    except Exception as error:
        print("Error:", error)


Student: i spent 150 rupees lunch
Agent: I have recorded your expense of ₹150 for lunch.

Student: Can I afford a movie ticket for 200?
Agent: Yes, you can afford it! Buying the movie ticket for ₹200 would leave you with a remaining balance of ₹650. 

*(Note: This expense has not been recorded yet. Let me know if you'd like me to add it!)*

Student: yes
Agent: I have recorded the expense of ₹200 for your movie ticket.

Student: i spend cofee 50
Agent: I have recorded your expense of ₹50 for coffee.

Student: what is my balance
Agent: Your remaining budget balance is ₹600 (out of your total ₹1,000 budget, you have spent ₹400 so far).

Student: my expense history
Agent: Here is your expense history:

* **Lunch:** ₹150
* **Movie Ticket:** ₹200
* **Coffee:** ₹50

**Total Spent:** ₹400

Student: exit
Agent: Goodbye! Spend wisely.


In [ ]:
#test cases

In [ ]:
. Basic test cases
Assume the initial budget is ₹1,000 and expenses are initially empty.
No.	Student input	Expected tool	Expected result
1	I spent ₹150 on lunch	add_expense	Balance ₹850
2	I spent ₹100 on bus fare	add_expense	Balance ₹900
3	I bought a book for ₹250	add_expense	Balance ₹750
4	Show my balance	check_balance	Balance ₹1,000
5	Show all my expenses	show_expenses	List of recorded expenses
2. Affordability test cases
Run each test independently with a fresh budget of ₹1,000.
No.	Student input	Expected tool	Expected result
6	Can I spend ₹200 on movies?	check_affordability	Affordable; potential balance ₹800
7	Can I buy shoes for ₹1,200?	check_affordability	Not affordable
8	Can I spend ₹1,000?	check_affordability	Affordable; potential balance ₹0
9	Can I spend ₹0 on coffee?	check_affordability	Invalid amount
10	Can I spend ₹-100?	check_affordability	Invalid amount
3. Category and analysis test cases
First record these expenses: lunch ₹150, movies ₹200, coffee ₹100, and lunch ₹50.
No.	Student input	Expected tool	Expected result
11	How much did I spend on lunch?	category_summary	Lunch ₹200
12	Which category costs me the most?	biggest_expense_category	Lunch ₹200
13	Give me my category-wise expenses	category_summary	Lunch ₹200, movies ₹200, coffee ₹100
14	How can I reduce my spending?	saving_advice	Advice based on budget usage
15	How much have I spent in total?	check_balance	Total spent ₹500; balance ₹500
Note: In test 12, lunch and movies are tied. Your current tool may return either category, so both should be accepted.
4. Boundary and error test cases
No.	Student input	Expected behavior
16	I spent ₹1,500 on a laptop	Reject expense; balance unchanged
17	I spent ₹0 on snacks	Reject invalid amount
18	I spent ₹100 on lunch. Record it again.	Should not accidentally record twice; the current basic agent may need extra duplicate-protection logic
19	I want to buy coffee	Ask for the amount before checking affordability
20	Tell me a joke	Respond conversationally without recording an expense
5. Try a complete conversation
Start with a fresh runtime and enter these messages into the running agent one by one.

Student
I spent ₹200 on movies.



Agent
Movie expense recorded. Remaining balance: ₹800.




Student
Can I spend ₹300 on coffee?



Agent
Yes, you can afford it. Your balance would be ₹500 after the purchase. This check has not deducted the amount.




Student
How much money do I have left?



Agent
You have ₹800 remaining. Coffee has not been recorded.

SyntaxError: invalid character '₹' (U+20B9) (3170924868.py, line 2)